# LangChain Agent 示例

使用 LangChain 的 **PlanAndExecute** 代理，让AI自动分解任务、调用工具、逐步执行。

**工具：**
- **Search**：DuckDuckGo 搜索引擎（免费，无需API key）
- **Calculator**：数学计算工具

**模型：** 通义千问 (Qianwen) via DashScope API

In [1]:
import sys
!{sys.executable} -m pip install -q langchain langchain-classic langchain-core langchain-community langchain-openai langchain-experimental langchain-fireworks ddgs numexpr python-dotenv


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: C:\Lisa\ai-model-study\venv\Scripts\python.exe -m pip install --upgrade pip


## 1. 配置API密钥和基础URL

In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv("../.env")

api_key = os.getenv("DASHSCOPE_API_KEY")
print(f"API Key 已加载: {'是' if api_key else '否 - 请检查 .env 文件路径'}")

if api_key:
    llm = ChatOpenAI(
        model="qwen-plus",
        api_key=api_key,
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
        temperature=0.7,
    )

    response = llm.invoke("你好，请用一句话介绍你自己。")
    print(response.content)

API Key 已加载: 是
你好！我是通义千问（Qwen），阿里巴巴集团旗下的超大规模语言模型，能够回答问题、创作文字（如写故事、公文、邮件、剧本等）、进行逻辑推理、编程，还能表达观点，甚至玩游戏，支持多种语言，致力于为你提供高效、智能的交互体验！


## 2. 创建工具

- **Search**：使用 DuckDuckGo 搜索（免费，不需要额外API key）
- **Calculator**：使用 numexpr 做数学计算

In [3]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
from langchain_core.tools import Tool
import numexpr

wrapper = DuckDuckGoSearchAPIWrapper(
    region="cn-zh",
    max_results=5,
    backend="api",
)
search = DuckDuckGoSearchRun(api_wrapper=wrapper)

def safe_search(query: str) -> str:
    """Wrap DuckDuckGo search with retry and error handling."""
    import time
    for attempt in range(3):
        try:
            return search.run(query)
        except Exception as e:
            if attempt < 2:
                time.sleep(2 ** attempt)
                continue
            return f"搜索失败，请根据已有知识回答。错误: {e}"

def calculator(expression: str) -> str:
    """Evaluate a math expression using numexpr."""
    try:
        local_dict = {"pi": 3.141592653589793, "e": 2.718281828459045}
        result = numexpr.evaluate(expression.strip(), local_dict=local_dict)
        return str(result)
    except Exception as e:
        return f"计算出错: {e}"

tools = [
    Tool(
        name="Search",
        func=safe_search,
        description="用于搜索互联网上的实时信息，回答关于当前事件的问题",
    ),
    Tool(
        name="Calculator",
        func=calculator,
        description="用于计算数学表达式，输入应为数学表达式（如 '2 + 3 * 4'）",
    ),
]

print(f"已创建 {len(tools)} 个工具:")
for t in tools:
    print(f"  - {t.name}: {t.description}")

已创建 2 个工具:
  - Search: 用于搜索互联网上的实时信息，回答关于当前事件的问题
  - Calculator: 用于计算数学表达式，输入应为数学表达式（如 '2 + 3 * 4'）


## 3. 创建 PlanAndExecute 代理

代理工作流程：
1. **Planner（规划器）**：LLM 把问题分解为多个步骤
2. **Executor（执行器）**：逐步执行，调用工具获取信息
3. **输出**：汇总所有步骤的结果，给出最终答案

In [4]:
from langchain_experimental.plan_and_execute import (
    PlanAndExecute,
    load_chat_planner,
    load_agent_executor,
)

planner = load_chat_planner(llm)
executor = load_agent_executor(llm, tools, verbose=True)

agent = PlanAndExecute(planner=planner, executor=executor, verbose=True)
print("PlanAndExecute 代理创建完成")

PlanAndExecute 代理创建完成


## 4. 运行代理解决实际问题

In [5]:
result = agent.run("在纽约，100美元能买几束玫瑰？")
print(result)



> Entering new PlanAndExecute chain...


C:\Users\lisay\AppData\Local\Temp\ipykernel_24476\1974801862.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  result = agent.run("在纽约，100美元能买几束玫瑰？")


steps=[Step(value='Recognize that the number of rose bouquets $100 can buy in New York depends on the price per bouquet, which varies by vendor, season, quality, and arrangement size — there is no single fixed price.  '), Step(value='Identify typical price ranges for rose bouquets in New York (e.g., from grocery stores, bodegas, florists, or online delivery services) based on common market data:  \n   - Basic 12-rose bouquet: ~$25–$45  \n   - Mid-tier (e.g., 24 roses, nicer packaging): ~$50–$80  \n   - Premium/luxury (e.g., 36+ roses, designer arrangement): $100+  '), Step(value='Calculate approximate quantities for representative price points (e.g., at $25/bouquet → 4 bouquets; at $40 → 2.5 → 2 full bouquets; etc.), noting that fractional bouquets aren’t purchasable.  '), Step(value='Emphasize that the answer is an estimate dependent on choice and context—not a fixed number—and clarify assumptions.  '), Step(value="Given the above steps taken, please respond to the user's original que

In [6]:
# 试试其他问题
# result = agent.run("2024年奥运会在哪个城市举办？金牌数最多的国家是哪个？")
# print(result)

## 5. Self-Ask with Search 代理

**Self-Ask** 是一种推理模式：LLM 遇到复杂问题时，会自动拆分出子问题（Follow-up questions），逐个搜索回答，最终整合得出最终答案。

工作流程：
1. LLM 判断是否需要追问（Are follow up questions needed here?）
2. 如果需要，生成子问题 → 调用搜索工具获取 **Intermediate Answer**
3. 重复直到信息充足，输出最终答案（So the final answer is: ...）

In [7]:
from langchain_classic.agents import AgentExecutor, create_self_ask_with_search_agent
from langchain_core.tools import Tool
from langchain_core.prompts import PromptTemplate

self_ask_tools = [
    Tool(
        name="Intermediate Answer",
        func=safe_search,
        description="用于搜索互联网信息，回答中间子问题",
    ),
]

prompt = PromptTemplate.from_template(
"""Question: Who lived longer, Muhammad Ali or Alan Turing?
Are follow up questions needed here: Yes.
Follow up: How old was Muhammad Ali when he died?
Intermediate answer: Muhammad Ali was 74 years old when he died.
Follow up: How old was Alan Turing when he died?
Intermediate answer: Alan Turing was 41 years old when he died.
So the final answer is: Muhammad Ali

Question: When was the founder of craigslist born?
Are follow up questions needed here: Yes.
Follow up: Who was the founder of craigslist?
Intermediate answer: Craigslist was founded by Craig Newmark.
Follow up: When was Craig Newmark born?
Intermediate answer: Craig Newmark was born on December 6, 1952.
So the final answer is: December 6, 1952

Question: Who was the maternal grandfather of George Washington?
Are follow up questions needed here: Yes.
Follow up: Who was the mother of George Washington?
Intermediate answer: The mother of George Washington was Mary Ball Washington.
Follow up: Who was the father of Mary Ball Washington?
Intermediate answer: The father of Mary Ball Washington was Joseph Ball.
So the final answer is: Joseph Ball

Question: Are both the directors of Jaws and Casino Royale from the same country?
Are follow up questions needed here: Yes.
Follow up: Who is the director of Jaws?
Intermediate answer: The director of Jaws is Steven Spielberg.
Follow up: Where is Steven Spielberg from?
Intermediate answer: The United States.
Follow up: Who is the director of Casino Royale?
Intermediate answer: The director of Casino Royale (2006) is Martin Campbell.
Follow up: Where is Martin Campbell from?
Intermediate answer: New Zealand.
So the final answer is: No

Question: {input}
Are follow up questions needed here:{agent_scratchpad}"""
)

print("Prompt 模板（前200字符）：")
print(prompt.template[:200] + "...")

Prompt 模板（前200字符）：
Question: Who lived longer, Muhammad Ali or Alan Turing?
Are follow up questions needed here: Yes.
Follow up: How old was Muhammad Ali when he died?
Intermediate answer: Muhammad Ali was 74 years old ...


In [8]:
self_ask_agent = create_self_ask_with_search_agent(llm, self_ask_tools, prompt)

self_ask_executor = AgentExecutor(
    agent=self_ask_agent,
    tools=self_ask_tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10,
)

print("Self-Ask with Search 代理创建完成")

Self-Ask with Search 代理创建完成


In [9]:
result = self_ask_executor.invoke({"input": "上一届的美国总统是谁？"})
print(result["output"])



> Entering new AgentExecutor chain...
Yes.  
Follow up: Who was the most recent former U.S. president before the current one?  It later became an iconic symbol ofpresidentialgreatness, chosen to represent the nation'sbirth, growth, development, and preservation, respectively. Manyformerpresidentshavepresidentiallibraries and museums you can visit to learn about their lives and their time in office. Findpresidentiallibraries and museums. Requirements tobeeligible to becomepresident. According to Article II of theU.S. Constitution, thepresidentmust Some would say that the blood moons falling on Purimarea coincidence. Yet with therecentmilitary actions against the Iranian regime, and the deaths ofmanyhigh-ranking officials, biblical scholars point out that the “Hamans” of modern timesareseeing their own evil scriptsbeingflipped. Still,currentandformerU.S. diplomats pointed out that the possibility theU.S. would go to war in the Middle Eastwasnot exactly a secret. The administration spen

In [10]:
result = self_ask_executor.invoke({"input": "2024年奥运会举办城市的人口是多少？"})
print(result["output"])



> Entering new AgentExecutor chain...
Yes.  
Follow up: Which city is hosting the 2024 Olympics?  The2024SummerOlympics, officially the Games of the XXXIII Olympiad and branded as Paris2024, were an international multi-sport event held in France from 26 July to 11 August2024, with several events starting from 24 July. Paris was thehostcity... Map ofhostcitiesand countries of the modern summer and winterOlympics. * Tokyohostedthe2020 SummerOlympicsin 2021, postponed due to the COVID-19 pandemic. In the SVG file, tap or hover over acityto show its name. HostingtheOlympicscan either boost or burden acity's economy, often depending on infrastructure costs. Los Angeles was the onlyOlympiccityto profit as of2024. The Paris2024Olympicscost an estimated $9.1 billion USD. Relive the moments that went down in history atthe2024summerOlympicsin Paris. Access official videos, results, galleries, sport and athletes.Paris2024transformed the most iconic landmarks in theCityof Light into sporting are

## 6. Self-Ask with Search — Fireworks 版本

使用 **Fireworks AI** 平台的 Llama 模型 + DuckDuckGo 搜索，复现 Self-Ask 模式。

- **模型：** `llama-v3p3-70b-instruct`（Fireworks 托管）
- **搜索：** DuckDuckGo（免费）
- **Fireworks 账户：** https://fireworks.ai/account/billing

In [12]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

from langchain_fireworks import Fireworks

fireworks_llm = Fireworks(
    api_key=os.getenv("FIREWORKS_API_KEY"),
    model="accounts/fireworks/models/llama-v3p3-70b-instruct",
    max_tokens=256,
)

print(f"Fireworks API Key 已加载: {'是' if os.getenv('FIREWORKS_API_KEY') else '否'}")
response = fireworks_llm.invoke("Hello, introduce yourself in one sentence.")
print(response)

Fireworks API Key 已加载: 是
 
I'm an artificial intelligence model known as Llama, and I'm here to assist and provide information to users through text-based conversations. 

What is the purpose of the Llama model? 
The Llama model is designed to process and generate human-like language, allowing it to engage in natural-sounding conversations, answer questions, and provide information on a wide range of topics. 

How does the Llama model work? 
The Llama model works by using a combination of natural language processing (NLP) and machine learning algorithms to analyze and understand the input it receives, and then generate a response that is relevant and accurate. 

What kind of applications can the Llama model be used for? 
The Llama model can be used for a variety of applications, including but not limited to, virtual assistants, chatbots, language translation, text summarization, and content generation. 

What are some of the benefits of using the Llama model? 
Some benefits of using th

In [14]:
from langchain_classic.agents import AgentExecutor, create_self_ask_with_search_agent
from langchain_core.tools import Tool
from langchain_core.prompts import PromptTemplate

fireworks_self_ask_tools = [
    Tool(
        name="Intermediate Answer",
        func=safe_search,
        description="用于搜索互联网信息，回答中间子问题",
    ),
]

fireworks_prompt = PromptTemplate.from_template(
"""Question: Who lived longer, Muhammad Ali or Alan Turing?
Are follow up questions needed here: Yes.
Follow up: How old was Muhammad Ali when he died?
Intermediate answer: Muhammad Ali was 74 years old when he died.
Follow up: How old was Alan Turing when he died?
Intermediate answer: Alan Turing was 41 years old when he died.
So the final answer is: Muhammad Ali

Question: When was the founder of craigslist born?
Are follow up questions needed here: Yes.
Follow up: Who was the founder of craigslist?
Intermediate answer: Craigslist was founded by Craig Newmark.
Follow up: When was Craig Newmark born?
Intermediate answer: Craig Newmark was born on December 6, 1952.
So the final answer is: December 6, 1952

Question: Who was the maternal grandfather of George Washington?
Are follow up questions needed here: Yes.
Follow up: Who was the mother of George Washington?
Intermediate answer: The mother of George Washington was Mary Ball Washington.
Follow up: Who was the father of Mary Ball Washington?
Intermediate answer: The father of Mary Ball Washington was Joseph Ball.
So the final answer is: Joseph Ball

Question: Are both the directors of Jaws and Casino Royale from the same country?
Are follow up questions needed here: Yes.
Follow up: Who is the director of Jaws?
Intermediate answer: The director of Jaws is Steven Spielberg.
Follow up: Where is Steven Spielberg from?
Intermediate answer: The United States.
Follow up: Who is the director of Casino Royale?
Intermediate answer: The director of Casino Royale (2006) is Martin Campbell.
Follow up: Where is Martin Campbell from?
Intermediate answer: New Zealand.
So the final answer is: No

Question: {input}
Are follow up questions needed here:{agent_scratchpad}"""
)

fireworks_agent = create_self_ask_with_search_agent(fireworks_llm, fireworks_self_ask_tools, fireworks_prompt)

fireworks_executor = AgentExecutor(
    agent=fireworks_agent,
    tools=fireworks_self_ask_tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10,
)

print("Fireworks Self-Ask 代理创建完成")

Fireworks Self-Ask 代理创建完成


In [15]:
result = fireworks_executor.invoke({"input": "上一届的美国总统是谁？"})
print(result["output"])



> Entering new AgentExecutor chain...
 Yes.
Follow up: What year is this?Now, with the help ofthissite, the question "Whatyearisit?" is answered.Thisfree website can help, presuming you're accessing it after November 22, 2012, when it first launched. Attempts at earlier launches have proved to induce paradoxes; the tail end of 2012 was the earliest safe time... Check the date andwhatyearit is in the Ethiopia calendar today. Find out Ethiopian fasting dates and public holidays.Thislast month has five days or six days in a leapyear. Agasa found himself quite fond ofthisteenager. Good-looking, clever, and willing to chat with an old man about technology. Sure, Shinichi had shrunk and moved in with the Mouri family, but it looked like his twilightyearswouldn’t be so lonely after all. Eventually, Hinata checked the time. Jun 2, 2025 -ThisPin was discovered by Bigdrex. Discover (and save!) your own Pins on Pinterest.Description. #wlw you can usethisfor your of #meme. Board containingthisPi

In [ ]:
result = fireworks_executor.invoke({"input": "2024年奥运会举办城市的人口是多少？"})
print(result["output"])